# Advanced Python – Web Scraping Foundations

Author: Elena Fuchs  
Course: Advanced Python/CIP (MSc Data Science)  
Focus: Static vs Dynamic scraping workflows

## Objective

This notebook explores foundational web scraping techniques using 
BeautifulSoup and Selenium. The goal is to understand when dynamic 
rendering is required and how to structure reproducible extraction workflows.

In [2]:
## Environment Setup

import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import pandas as pd

These libraries allow:
- HTTP requests
- HTML parsing
- Browser automation
- Data structuring

## 1. Static Web Scraping (BeautifulSoup)

Used when:
- Content is present in raw HTML
- No JavaScript rendering is required

In [ ]:
url = "https://example.com"
response = requests.get(url)

soup = BeautifulSoup(response.text, "html.parser")
elements = soup.select("div.product")

### Key Insight
BeautifulSoup parses server-returned HTML.
If content is not in `response.text`, it likely requires JavaScript rendering.

## 2. Dynamic Web Scraping (Selenium)

Required when:
- Content is loaded via JavaScript
- API calls are triggered after page load

In [ ]:
driver = webdriver.Chrome()
driver.get(url)

time.sleep(5)
page = driver.page_source
driver.quit()

soup = BeautifulSoup(page, "html.parser")

### Engineering Considerations
- Always close the driver
- Avoid unnecessary sleep delays (use WebDriverWait in production)
- Be mindful of ethical scraping and robots.txt

## 3. Extracting Repeated Elements

Websites often repeat structural blocks such as: 

&lt;div class="item"&gt;...&lt;/div&gt;

These can be extracted using CSS selectors.

In [ ]:
items = soup.select("div.item")
data = [item.text.strip() for item in items]

## 4. Structuring Data with pandas

In [ ]:
df = pd.DataFrame(data, columns=["item"])
df.head()

## Workflow Summary

1. Inspect HTML structure.
2. Determine static vs dynamic.
3. Choose appropriate tool.
4. Extract repeated elements.
5. Convert to structured format.
6. Clean and validate data.

## 5. Legal & Ethical Considerations

Web scraping must always be conducted responsibly and in accordance with legal and ethical standards.

### 1. Swiss Data Protection Law

In Switzerland, data processing is regulated under the **New Federal Act on Data Protection (nFADP)**, in effect since September 1, 2023.

The nFADP governs:
- The processing of personal data
- Transparency obligations
- Data security requirements

Official source:
https://www.kmu.admin.ch/kmu/en/home/facts-and-trends/digitization/data-protection/new-federal-act-on-data-protection-nfadp.html

Scraping personal data without legal basis may violate data protection law.

### 2. robots.txt Convention

Websites may specify crawling permissions via a `robots.txt` file.

- Pages explicitly disallowed must not be scraped.
- Pages without restrictions are commonly considered accessible.
- The convention is widely supported (e.g., Google crawler standards).

Reference:
https://developers.google.com/search/docs/crawling-indexing/robots/intro

Note: `robots.txt` is not a law but an industry standard that should be respected.

### 3. Avoid Personal Data

As a general rule:

- Do not scrape personalized or identifiable data.
- Avoid collecting user profiles, private information, or sensitive content.
- Focus on publicly available, non-personal data whenever possible.

## Ethical Summary

- Scraping should not overload servers.
- Avoid excessive request frequency.
- Use rate limiting and respectful delays.
- Prefer official APIs when available.

**Principle: Be respectful to websites and their infrastructure.**

## 6. Exercise: Prefer APIs over Scraping (yfinance)

If data is available through an API/library, it's often more reliable than scraping HTML.
This cell demonstrates pulling quote information from Yahoo Finance via `yfinance`.

In [6]:
!pip -q install yfinance

import yfinance as yf

ticker = yf.Ticker("GOOG")

# current-ish price (depends on market state)
price = ticker.fast_info.get("last_price") or ticker.info.get("regularMarketPrice")
name  = ticker.info.get("shortName")

print("Name:", name)
print("Price:", price)

# some “quote statistics” style info
keys = [
    "marketCap", "trailingPE", "forwardPE", "dividendYield",
    "fiftyTwoWeekLow", "fiftyTwoWeekHigh", "beta",
]
for k in keys:
    print(f"{k:20}:", ticker.info.get(k))

Name: Alphabet Inc.
Price: 303.56
marketCap           : 3672165318656
trailingPE          : 28.081406
forwardPE           : 22.710627
dividendYield       : 0.28
fiftyTwoWeekLow     : 142.66
fiftyTwoWeekHigh    : 350.15
beta                : 1.086


#populating own database
- you can scrape whenever you want e.g. stock values
- you are exposed to changes, often have to adjust your script

## 7. Dynamic Scraping Example – Kaggle (JavaScript-rendered Content)

Kaggle dataset search results are dynamically rendered via JavaScript.
Therefore, static requests (`requests.get`) will not return the full content.

We use Selenium to render the page in a real browser instance and extract the loaded HTML.

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import time

# Target URL
url = "https://www.kaggle.com/datasets?search=Data+Visualization"

# Initialize WebDriver
driver = webdriver.Chrome()

try:
    driver.get(url)

    # Allow JavaScript to load content
    time.sleep(5)  # For production: use WebDriverWait instead

    # Extract fully rendered HTML
    page_source = driver.page_source
    soup = BeautifulSoup(page_source, "html.parser")

    # Extract dataset list items using CSS selector
    selector = "li.MuiListItem-root"
    items = soup.select(selector)

    dataset_titles = [item.get_text(strip=True) for item in items]

    print("Extracted items:")
    for title in dataset_titles:
        print("-", title)

finally:
    driver.quit()

### Observations

- Kaggle uses JavaScript rendering.
- Static requests return incomplete HTML.
- Selenium successfully retrieves rendered content.
- CSS selectors should be kept minimal and robust.

## Key Takeaways

- Always inspect page source before choosing tools.
- Use CSS selectors for scalable extraction.
- Selenium simulates a browser but is heavier.
- Structured data pipelines matter more than raw scraping.